# 06 - Eksperimen tanpa `code_gender`

**Pertanyaan:** `code_gender` masuk 5 fitur terpenting di SHAP. Kalau kolom ini dibuang, seberapa turun performa model?

**Desain:** parameter, split, dan seed sama persis dengan model final (`models/best_params.json`). Satu-satunya perbedaan adalah kolom `code_gender`. Set test hanya dipakai untuk evaluasi akhir.

In [1]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yml").exists())
sys.path.append(str(ROOT))
from src.data import load_model_table, prepare, split

warnings.filterwarnings("ignore", message=".*eval_set.*")
sns.set_theme(style="whitegrid")

X, y = prepare(load_model_table())
X_train, X_valid, X_test, y_train, y_valid, y_test = split(X, y)
params = json.loads((ROOT / "models" / "best_params.json").read_text(encoding="utf-8"))["best_params"]
print(X_train.shape, "| kolom gender ada:", "code_gender" in X.columns)
print(X["code_gender"].value_counts(dropna=False))

(215257, 165) | kolom gender ada: True
code_gender
F      202448
M      105059
XNA         4
Name: count, dtype: int64


## 1. Latih ulang dengan dan tanpa `code_gender`

In [2]:
BASE = dict(n_estimators=2000, subsample_freq=1, metric="auc", random_state=42, n_jobs=-1, verbose=-1)

def fit(drop_cols):
    cols = [c for c in X_train.columns if c not in drop_cols]
    model = lgb.LGBMClassifier(**BASE, **params)
    model.fit(X_train[cols], y_train, eval_set=[(X_valid[cols], y_valid)],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    return model, cols

with_gender, cols_with = fit([])
no_gender, cols_no = fit(["code_gender"])

proba = {
    "dengan code_gender": with_gender.predict_proba(X_test[cols_with])[:, 1],
    "tanpa code_gender": no_gender.predict_proba(X_test[cols_no])[:, 1],
}
rows = []
for name, p in proba.items():
    rows.append({"model": name,
                 "pohon": (with_gender if "dengan" in name else no_gender).best_iteration_,
                 "roc_auc": round(roc_auc_score(y_test, p), 4),
                 "pr_auc": round(average_precision_score(y_test, p), 4),
                 "brier": round(brier_score_loss(y_test, p), 4)})
compare = pd.DataFrame(rows)
compare

,model,pohon,roc_auc,pr_auc,brier
0,dengan code_gender,1253,0.7913,0.2845,0.0657
1,tanpa code_gender,1140,0.7893,0.2822,0.0659


## 2. Apakah selisihnya lebih besar dari noise?

Bootstrap berpasangan pada set test: ambil sampel ulang pemohon yang sama untuk kedua model, hitung selisih ROC-AUC.

In [3]:
rng = np.random.default_rng(42)
yt = y_test.values
pa, pb = proba["dengan code_gender"], proba["tanpa code_gender"]
diffs = []
for _ in range(300):
    idx = rng.integers(0, len(yt), len(yt))
    if yt[idx].min() == yt[idx].max():
        continue
    diffs.append(roc_auc_score(yt[idx], pa[idx]) - roc_auc_score(yt[idx], pb[idx]))
diffs = np.array(diffs)
print(f"Selisih ROC-AUC (dengan - tanpa): {roc_auc_score(yt, pa) - roc_auc_score(yt, pb):+.4f}")
print(f"Interval 95% bootstrap: [{np.percentile(diffs, 2.5):+.4f}, {np.percentile(diffs, 97.5):+.4f}]")

Selisih ROC-AUC (dengan - tanpa): +0.0019
Interval 95% bootstrap: [+0.0004, +0.0033]


## 3. Dampak ke kebijakan persetujuan (ambang p < 0,143)

Bandingkan siapa yang disetujui oleh dua model, dan bagaimana tingkat persetujuan per gender berubah.

In [4]:
THRESHOLD = 0.143
d = pd.DataFrame({"gender": X_test["code_gender"].astype(str).values, "y": yt,
                  "approve_with": pa < THRESHOLD, "approve_no": pb < THRESHOLD})

def summarize(col):
    a = d[d[col]]
    return pd.Series({"disetujui_%": round(d[col].mean() * 100, 1),
                      "default_rate_disetujui_%": round(a["y"].mean() * 100, 2)})

policy = pd.DataFrame({"dengan code_gender": summarize("approve_with"),
                       "tanpa code_gender": summarize("approve_no")}).T
display(policy)

by_gender = d.groupby("gender").agg(
    pemohon=("y", "size"),
    default_rate_aktual=("y", lambda s: round(s.mean() * 100, 2)),
    setuju_dengan=("approve_with", lambda s: round(s.mean() * 100, 1)),
    setuju_tanpa=("approve_no", lambda s: round(s.mean() * 100, 1)),
)
by_gender

,disetujui_%,default_rate_disetujui_%
dengan code_gender,84.7,4.85
tanpa code_gender,84.8,4.90


,pemohon,default_rate_aktual,setuju_dengan,setuju_tanpa
gender,,,,
F,30429,6.96,88.0,87.2
M,15698,10.23,78.3,80.0


## 4. Catatan temuan
Menghapus code_gender hanya menurunkan ROC-AUC 0,002 (0,791 → 0,789) dan tidak mengubah hasil kebijakan secara berarti, sehingga model tidak perlu bergantung pada atribut sensitif ini. Selisih persetujuan antar gender menyempit tetapi tidak hilang, karena fitur lain bisa menjadi proxy; audit fairness lebih lanjut tetap diperlukan.”